## 1. Setup

In [ ]:
# Cell 1 — install, imports, mount Drive
!pip -q install torch_geometric
from google.colab import drive
drive.mount('/content/drive')

import os, glob, re, zipfile, numpy as np, pandas as pd
import torch, torch.nn as nn, matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from torch_geometric.data import Data, Batch
from torch_geometric.nn import EdgeConv

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Configuration

In [ ]:
# Cell 2 — paths and constants (edit ZIP_PATH to where your dataset lives)
ZIP_PATH   = '/content/drive/MyDrive/simjeb_gnn/SimJEB/dataverse_files.zip'
WORK_DIR   = '/content/simjeb'
FIELD_DIR  = '/content/fields'
CACHE      = '/content/drive/MyDrive/simjeb_gnn/surface_dataset.npz'
MODEL_PATH = '/content/drive/MyDrive/simjeb_gnn/gnn_model.pt'

N_NODES = 8000                     # surface points per bracket
K       = 12                       # neighbours per node
LOADS   = ['ver', 'hor', 'dia', 'tor']
STRESS  = [f'{l}_stress' for l in LOADS]
LOAD_NAMES = ['vertical', 'horizontal', 'diagonal', 'torsional']

## 3. Build the dataset

In [ ]:
# Cell 3 — extract SimJEB and build the surface-point dataset (positions + stress)
os.makedirs(WORK_DIR, exist_ok=True)
if not glob.glob(f'{WORK_DIR}/**/all_bracket_metadata.csv', recursive=True):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extract('all_bracket_metadata.csv', WORK_DIR)
META_PATH = glob.glob(f'{WORK_DIR}/**/all_bracket_metadata.csv', recursive=True)[0]

os.makedirs(FIELD_DIR, exist_ok=True)
if len(glob.glob(f'{FIELD_DIR}/**/*field*.csv', recursive=True)) < 300:
    with zipfile.ZipFile(ZIP_PATH) as z:
        for n in [x for x in z.namelist() if 'simresults_(csv)' in x.lower() and x.endswith('.zip')]:
            z.extract(n, '/content/_tmp')
            with zipfile.ZipFile('/content/_tmp/' + n) as inner:
                inner.extractall(FIELD_DIR)

if os.path.exists(CACHE):
    d = np.load(CACHE); POS, STR, IS_TEST, IDS = d['pos'], d['stress'], d['is_test'], d['ids']
else:
    ids, pos_list, str_list = [], [], []
    for f in sorted(glob.glob(f'{FIELD_DIR}/**/*field*.csv', recursive=True)):
        m = re.search(r'(\d+)field', os.path.basename(f))
        if not m: continue
        df = pd.read_csv(f, usecols=['surf','x','y','z'] + STRESS)
        df = df[df['surf'] == 1]                       # surface nodes only
        if len(df) < N_NODES: continue
        df = df.sample(N_NODES, random_state=0)        # downsample for speed
        p = df[['x','y','z']].to_numpy(np.float32)
        p = (p - p.mean(0)) / (p.std() + 1e-6)         # normalize each bracket
        ids.append(int(m.group(1))); pos_list.append(p)
        str_list.append(df[STRESS].to_numpy(np.float32))
    IDS, POS, STR = np.array(ids), np.stack(pos_list), np.stack(str_list)
    meta0 = pd.read_csv(META_PATH).set_index('id')
    IS_TEST = np.array([bool(meta0.loc[i, 'test_split_0']) for i in IDS])
    np.savez_compressed(CACHE, ids=IDS, pos=POS, stress=STR, is_test=IS_TEST)

print('brackets:', POS.shape[0], '| train:', int((~IS_TEST).sum()), '| test:', int(IS_TEST.sum()))

## 4. Baseline (geometry-blind random forest)

In [ ]:
# Cell 4 — classical baseline: predict MAX stress from global descriptors only
meta = pd.read_csv(META_PATH)
feat_cols = ['volume','surface_area','average_edge_length','genus','num_tets','mass','num_vertices','num_faces']
targ_cols = [f'max_{l}_stress' for l in LOADS]
m = meta.dropna(subset=feat_cols + targ_cols + ['test_split_0'])

Xtr = m.loc[~m.test_split_0, feat_cols].values
Xte = m.loc[ m.test_split_0, feat_cols].values
ytr = np.log10(m.loc[~m.test_split_0, targ_cols].clip(lower=1e-6).values)
yte = m.loc[ m.test_split_0, targ_cols].values

rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1).fit(Xtr, ytr)
pred = 10 ** rf.predict(Xte)
print('Baseline R2 (max stress):')
for i, nm in enumerate(LOAD_NAMES):
    print(f'  {nm:<11}{r2_score(yte[:, i], pred[:, i]):.3f}')

 ## 5. Turn each bracket into a graph

In [ ]:
# Cell 5 — build graphs: node features = position + normal + size; kNN edges; stress targets
meta = pd.read_csv(META_PATH).set_index('id')
G = np.log1p(meta.loc[IDS, ['volume','mass','surface_area']].to_numpy(np.float32))
G = (G - G[~IS_TEST].mean(0)) / (G[~IS_TEST].std(0) + 1e-6)

y_train = np.log1p(STR[~IS_TEST]).reshape(-1, 4)
Y_MEAN, Y_STD = y_train.mean(0), y_train.std(0)
encode = lambda s: (np.log1p(s) - Y_MEAN) / Y_STD
decode = lambda z: np.expm1(z * Y_STD + Y_MEAN)

def build_graph(i):
    pos = POS[i]
    nbr = NearestNeighbors(n_neighbors=K+1).fit(pos).kneighbors(pos, return_distance=False)[:, 1:]
    loc = pos[nbr] - pos[nbr].mean(1, keepdims=True)          # local neighbourhood
    cov = np.einsum('nki,nkj->nij', loc, loc) / K
    normal = np.linalg.eigh(cov)[1][:, :, 0]                  # surface normal via PCA
    s = np.sign(np.einsum('ni,ni->n', normal, pos)); s[s == 0] = 1
    normal = (normal * s[:, None]).astype(np.float32)         # orient outward
    x = np.concatenate([pos, normal, np.tile(G[i], (N_NODES, 1))], 1)   # 9 features
    src = np.repeat(np.arange(N_NODES), K); dst = nbr.reshape(-1)
    edges = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])])
    return Data(x=torch.tensor(x), edge_index=torch.tensor(edges, dtype=torch.long),
                y=torch.tensor(encode(STR[i])))

graphs = [build_graph(i) for i in range(POS.shape[0])]
train_graphs = [graphs[i] for i in range(len(graphs)) if not IS_TEST[i]]
test_graphs  = [graphs[i] for i in range(len(graphs)) if IS_TEST[i]]
print('graphs built:', len(graphs))

## 6. The GNN model

In [ ]:
# Cell 6 — EdgeConv (DGCNN-style) network: per-node stress for 4 load cases
def mlp(i, o):
    return nn.Sequential(nn.Linear(i, o), nn.ReLU(), nn.Linear(o, o), nn.ReLU())

class StressGNN(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.conv1 = EdgeConv(mlp(2*9, h))            # 9 input features
        self.conv2 = EdgeConv(mlp(2*h, h))
        self.conv3 = EdgeConv(mlp(2*h, h))
        self.conv4 = EdgeConv(mlp(2*h, h))
        self.head  = nn.Sequential(nn.Linear(4*h, h), nn.ReLU(), nn.Linear(h, 4))
    def forward(self, x, edge_index):
        a = self.conv1(x, edge_index)
        b = self.conv2(a, edge_index)
        c = self.conv3(b, edge_index)
        d = self.conv4(c, edge_index)
        return self.head(torch.cat([a, b, c, d], dim=1))   # multi-scale features

## 7. Train

In [ ]:
# Cell 7 — train with MSE loss, Adam, cosine schedule
torch.manual_seed(0)
model = StressGNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS, BATCH = 200, 6
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

def batched(gs, bs, shuffle=False):
    order = np.random.permutation(len(gs)) if shuffle else np.arange(len(gs))
    for k in range(0, len(gs), bs):
        yield Batch.from_data_list([gs[j] for j in order[k:k+bs]]).to(device)

for ep in range(EPOCHS):
    model.train(); total = 0
    for batch in batched(train_graphs, BATCH, shuffle=True):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(batch.x, batch.edge_index), batch.y)
        loss.backward(); opt.step(); total += loss.item()
    sched.step()
    if (ep + 1) % 40 == 0:
        print(f'epoch {ep+1}/{EPOCHS}  MSE {total/len(train_graphs)*BATCH:.4f}')

torch.save(model.state_dict(), MODEL_PATH)
print('saved model')

## 8. Evaluate (train vs test)

In [ ]:
# Cell 8 — metrics on train and test; the gap shows overfitting
model.eval()
def predict(gs):
    out = []
    with torch.no_grad():
        for batch in batched(gs, BATCH):
            out.append(model(batch.x, batch.edge_index).cpu().numpy())
    return decode(np.concatenate(out))

def evaluate(gs, label):
    P = predict(gs); T = decode(np.concatenate([g.y.numpy() for g in gs]))
    n = len(gs); Pb, Tb = P.reshape(n, N_NODES, 4), T.reshape(n, N_NODES, 4)
    print(f'\n=== {label} ({n} brackets) ===')
    print(f'{"load":<11}{"R2":>8}{"MAE":>9}{"corr":>8}')
    for j, nm in enumerate(LOAD_NAMES):
        corr = np.median([np.corrcoef(Pb[b,:,j], Tb[b,:,j])[0,1] for b in range(n)])
        print(f'{nm:<11}{r2_score(T[:,j],P[:,j]):>8.3f}{mean_absolute_error(T[:,j],P[:,j]):>9.1f}{corr:>8.3f}')
    print(f'{"OVERALL":<11}{r2_score(T, P):>8.3f}')

evaluate(train_graphs, 'TRAIN')
evaluate(test_graphs,  'TEST')

## 9. Visualize predicted vs true stress

In [ ]:
# Cell 9 — gallery of predicted vs FEA stress fields (change LC for other loads)
LC = 3   # 0 vertical, 1 horizontal, 2 diagonal, 3 torsional
P = predict(test_graphs).reshape(len(test_graphs), N_NODES, 4)
T = decode(np.concatenate([g.y.numpy() for g in test_graphs])).reshape(len(test_graphs), N_NODES, 4)
test_pos = [POS[i] for i in range(POS.shape[0]) if IS_TEST[i]]
test_ids = IDS[IS_TEST]

ranking = sorted([(np.corrcoef(P[b,:,LC], T[b,:,LC])[0,1], b) for b in range(len(test_graphs))], reverse=True)
pick = [ranking[k][1] for k in np.linspace(0, len(ranking)-1, 6).astype(int)]

fig = plt.figure(figsize=(9, 26))
for row, b in enumerate(pick):
    t, p, xyz = T[b,:,LC], P[b,:,LC], test_pos[b]; r = np.corrcoef(p, t)[0,1]
    vmin, vmax = float(t.min()), float(np.percentile(t, 99))
    for col, (vals, lab) in enumerate([(t, 'True (FEA)'), (p, 'GNN')]):
        ax = fig.add_subplot(6, 2, row*2+col+1, projection='3d')
        ax.scatter(xyz[:,0], xyz[:,1], xyz[:,2], c=vals, cmap='turbo', vmin=vmin, vmax=vmax, s=2)
        ax.set_title(f'#{test_ids[b]}  {lab}  (r={r:.2f})', fontsize=8); ax.set_axis_off(); ax.view_init(20, 120)
plt.suptitle(f'Predicted vs FEA stress — {LOAD_NAMES[LC]} load', y=1.0); plt.tight_layout()
plt.savefig('/content/drive/MyDrive/simjeb_gnn/stress_field_compare.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10 — solid-surface (mesh-look) comparison images with Plotly
!pip -q install -U kaleido
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree

meta = pd.read_csv(META_PATH).set_index('id')          # ensure id-indexed
gf = np.log1p(meta.loc[IDS, ['volume','mass','surface_area']].to_numpy(np.float32))
GMEAN, GSTD = gf[~IS_TEST].mean(0), gf[~IS_TEST].std(0) + 1e-6

ALPHA_FACTOR = 3      # bigger = smoother/fills holes; smaller = more detail/can fragment
MAX_VIZ_NODES = 15000 # cap points per bracket for fast, clean rendering

def predict_full(bid):
    f = glob.glob(f'{FIELD_DIR}/**/{bid}field.csv', recursive=True)[0]
    df = pd.read_csv(f, usecols=['surf','x','y','z'] + STRESS)
    df = df[df['surf'] == 1]
    if len(df) > MAX_VIZ_NODES:
        df = df.sample(MAX_VIZ_NODES, random_state=0)
    xyz = df[['x','y','z']].to_numpy(np.float32)        # real mm coordinates (for display)
    true = df[STRESS].to_numpy(np.float32)
    pos = (xyz - xyz.mean(0)) / (xyz.std() + 1e-6)      # normalize like training
    nbr = NearestNeighbors(n_neighbors=K+1).fit(pos).kneighbors(pos, return_distance=False)[:, 1:]
    loc = pos[nbr] - pos[nbr].mean(1, keepdims=True)
    nrm = np.linalg.eigh(np.einsum('nki,nkj->nij', loc, loc) / K)[1][:, :, 0]
    s = np.sign(np.einsum('ni,ni->n', nrm, pos)); s[s == 0] = 1
    nrm = (nrm * s[:, None]).astype(np.float32)
    g = (np.log1p(meta.loc[bid, ['volume','mass','surface_area']].to_numpy(np.float32)) - GMEAN) / GSTD
    x = np.concatenate([pos, nrm, np.tile(g, (len(pos), 1))], 1)
    src = np.repeat(np.arange(len(pos)), K); dst = nbr.reshape(-1)
    ei = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])])
    model.eval()
    with torch.no_grad():
        pred = decode(model(torch.tensor(x).to(device),
                            torch.tensor(ei, dtype=torch.long).to(device)).cpu().numpy())
    return xyz, true, pred

def surface_image(bid, LC=3, mode='surface'):
    xyz, true, pred = predict_full(bid)
    t, p = true[:, LC], pred[:, LC]
    cmin, cmax = float(t.min()), float(np.percentile(t, 99))
    alpha = ALPHA_FACTOR * np.median(cKDTree(xyz).query(xyz, k=2)[0][:, 1])
    fig = make_subplots(rows=1, cols=2, specs=[[{'type':'scene'}, {'type':'scene'}]],
                        subplot_titles=[f'#{bid} True (FEA)', f'#{bid} GNN prediction'],
                        horizontal_spacing=0.02)
    for col, vals in enumerate([t, p], start=1):
        if mode == 'surface':
            fig.add_trace(go.Mesh3d(x=xyz[:,0], y=xyz[:,1], z=xyz[:,2], intensity=vals,
                          alphahull=float(alpha), colorscale='Turbo', cmin=cmin, cmax=cmax,
                          showscale=(col==2), colorbar=dict(title='MPa', len=0.6)), row=1, col=col)
        else:
            fig.add_trace(go.Scatter3d(x=xyz[:,0], y=xyz[:,1], z=xyz[:,2], mode='markers',
                          marker=dict(size=2.5, color=vals, colorscale='Turbo', cmin=cmin, cmax=cmax,
                          showscale=(col==2), colorbar=dict(title='MPa', len=0.6))), row=1, col=col)
    cam = dict(eye=dict(x=1.4, y=1.4, z=1.0))
    fig.update_layout(width=1100, height=550, title=f'{LOAD_NAMES[LC]} load',
                      scene=dict(aspectmode='data', camera=cam),
                      scene2=dict(aspectmode='data', camera=cam),
                      margin=dict(l=0, r=0, t=60, b=0))
    return fig

# pick a strong and a median test bracket (ranked on the 8k predictions we already have)
P8 = predict(test_graphs).reshape(len(test_graphs), N_NODES, 4)
T8 = decode(np.concatenate([g.y.numpy() for g in test_graphs])).reshape(len(test_graphs), N_NODES, 4)
test_ids = IDS[IS_TEST]; LC = 3
rank = sorted([(np.corrcoef(P8[b,:,LC], T8[b,:,LC])[0,1], int(test_ids[b])) for b in range(len(test_graphs))], reverse=True)
chosen = [rank[0][1], rank[len(rank)//2][1]]

for bid in chosen:
    fig = surface_image(bid, LC=LC, mode='surface')   # use mode='points' if the surface looks rough
    fig.show()
    try: fig.write_image(f'/content/drive/MyDrive/simjeb_gnn/surface_{bid}.png', scale=2)
    except Exception as e: print('PNG export skipped:', e)